#Pivoting

In [0]:
df_csv=spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv")

display(df_csv)

In [0]:
from pyspark.sql.functions import count
df_pivot=df_csv.groupBy("customer_id").pivot("order_status").agg(count("order_id"))

display(df_pivot)

#When Otherwise

In [0]:
from pyspark.sql.functions import when,col
df_ifelse=df_pivot.withColumn("return_flag",when(col("Returned")==1,"Low")\
        .when(col("Returned")<=3,"Medium")\
        .when(col("Returned")>3,"High")\
        .otherwise("No Return")
)

display(df_ifelse)

In [0]:
df_ifelse_new=df_pivot.withColumn("return_flag",when(col("Returned")==1,"Low")\
    .when((col("Returned")==2) & (col("Returned")==3),"Medium")\
    .when(col("Returned")>4,"High")\
    .otherwise("No Return")
    )

display(df_ifelse_new)

#Joins

In [0]:
#Dummy Data
data=[
    (1,"abc","123"),
    (2,"def","456"),
    (3,"ghi","789"),
    (4,"jkl","101112"),
    (5,"mno","131415"),
    (6,"pqr","161718"),
    (7,"stu","192021"),
    (8,"vwx","222324"),
    (9,"yz","252627"),
    (10,"abc","123"),
    (11,"def","456"),
    (12,"ghi","789"),
    (13,"jkl","101112"),
    (14,"mno","131415"),
    (15,"pqr","161718"),
    (16,"stu","192021"),
    (17,"vwx","222324"),
    (18,"yz","252627"),
    (19,"abc","123"),
    (20,"def","456"),
    (21,"ghi","789"),
    (22,"jkl","101112"),
    (23,"mno","131415"),
    (24,"pqr","161718"),
    (25,"stu","192021"),
    (26,"vwx","222324")
]

schema=["id","name","code"]

df1=spark.createDataFrame(data,schema)


data2=[
    (101,1),
    (102,2),
    (103,3),
    (104,4),
    (105,5),
    (106,6),
    (107,7),
    (108,8),
    (109,9),
    (110,10),
    (111,11),
    (112,12),
    (113,13),
    (114,14),
    (115,15),
    (116,31),
    (117,32),
    (118,33),
    (119,34),
    (120,35),
    (121,36),
    (122,37),
    (123,38),
    (124,39),
    (125,40)
]

schema2=["order_id","id"]

df2=spark.createDataFrame(data2,schema2)


display(df1)
display(df2)

###Inner Join

In [0]:
df_inner=df1.join(df2,df1["id"]==df2["id"],"inner")

display(df_inner)

###Left Join

In [0]:
df_left=df1.join(df2,df1["id"]==df2["id"],"left")

display(df_left)

###Right Join

In [0]:
df_right=df1.join(df2,df1["id"]==df2["id"],"right")

display(df_right)

###Full Join

In [0]:
df_full=df1.join(df2,df1["id"]==df2["id"],"full")

display(df_full)


                         


###Anti Join

In [0]:
df_anti=df1.join(df2,df1["id"]==df2["id"],"anti")

display(df_anti)

#Window Functions

In [0]:
data=[
    (35),
    (45),
    (99),
    (99),
    (100)
]

schema="marks int"

df=spark.createDataFrame(data=data,schema=schema)

display(df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df_row = df.withColumn(
    "marks_seq",
    row_number().over(Window.orderBy("marks"))
)

display(df_row)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df_row=df.withColumn(
    "marks_seq",
    row_number().over(Window.orderBy("marks"))
)
                     
display(df_row)

In [0]:
df_rank=df.withColumn(
    "marks_rank",
    row_number().over(Window.partitionBy("marks").orderBy("marks"))
)

display(df_rank)

####Rank and Dense_rank

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank,dense_rank

df_ranking=(
    df.withColumn("rank",
                  rank().over(Window.orderBy("marks")))
        .withColumn("dense_rank",
                    dense_rank().over(Window.orderBy("marks")))
)

display(df_ranking)

#Special Functions

###Total Revenue

In [0]:
data=[
    (2020,100),
    (2021,110),
    (2022,120),
    (2023,130),
    (2024,150)
]

schema="year int, revenue int"

df=spark.createDataFrame(data,schema)

display(df)

In [0]:
 from pyspark.sql.functions import sum

df_total=(
    df.withColumn("total_revenue",
                  sum("revenue").over(Window.orderBy("year")))
)

display(df_total)

In [0]:
 from pyspark.sql.functions import sum

df_total=(
    df.withColumn("total_revenue",
                  sum("revenue").over(Window.orderBy("year")
                                      .rowsBetween(Window.unboundedPreceding,Window.currentRow)))
)

display(df_total)

In [0]:
df_total=(
    df.withColumn("total_revenue",
                  sum("revenue").over(Window.orderBy("year")
                                      .rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing)))
)

display(df_total)